In [42]:
# Import required libraries
import kagglehub
import os
import pandas as pd

In [43]:
# Download the Brazilian E-Commerce dataset from Kaggle
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

print("Path to dataset files:", path)

Path to dataset files: /Users/hilal/.cache/kagglehub/datasets/olistbr/brazilian-ecommerce/versions/2


In [44]:
# List all files in the downloaded dataset directory
files = os.listdir(path)

# Print each file name to see what datasets are available
print("Files in dataset:")
for file in files:
    print(file)

Files in dataset:
olist_sellers_dataset.csv
product_category_name_translation.csv
olist_orders_dataset.csv
olist_order_items_dataset.csv
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_products_dataset.csv


In [45]:
# Filter and extract only CSV files from the dataset directory
csv_files = [f for f in os.listdir(path) if f.endswith(".csv")]

# Initialize a dictionary to store all datasets
datasets = {}

# Load each CSV file into a pandas DataFrame and store in the dictionary
for file in csv_files:
    name = file.replace(".csv", "")  # Remove .csv extension to use as key
    datasets[name] = pd.read_csv(os.path.join(path, file))

# Display all loaded dataset names
print(datasets.keys())

dict_keys(['olist_sellers_dataset', 'product_category_name_translation', 'olist_orders_dataset', 'olist_order_items_dataset', 'olist_customers_dataset', 'olist_geolocation_dataset', 'olist_order_payments_dataset', 'olist_order_reviews_dataset', 'olist_products_dataset'])


In [46]:
# Iterate through all datasets and display their column names
for name, df in datasets.items():
    print(f"\nDataset: {name}")
    # Print all column names for each dataset to understand data structure
    print(df.columns.tolist())


Dataset: olist_sellers_dataset
['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']

Dataset: product_category_name_translation
['product_category_name', 'product_category_name_english']

Dataset: olist_orders_dataset
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

Dataset: olist_order_items_dataset
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

Dataset: olist_customers_dataset
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Dataset: olist_geolocation_dataset
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

Dataset: olist_order_payments_dataset
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

Dataset: olist_o

In [47]:
#Renaming the columns in olist_products_dataset for better readability
datasets["olist_products_dataset"].rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length"
}, inplace=True)

datasets["olist_products_dataset"].columns

Index(['product_id', 'product_category_name', 'product_name_length',
       'product_description_length', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='str')

In [48]:
# Analyze duplicates in key identifier columns for each dataset
# This helps identify data quality issues and potential errors
print("=" * 70)
print("DUPLICATE ANALYSIS FOR EACH DATASET")
print("=" * 70)

# Map each dataset to its primary key column for duplicate checking
key_columns = {
    "olist_orders_dataset": "order_id",
    "olist_customers_dataset": "customer_id",
    "olist_products_dataset": "product_id",
    "olist_order_reviews_dataset": "review_id",
    "olist_sellers_dataset": "seller_id",
    "olist_geolocation_dataset": "zip_code_prefix",
    "product_category_name_translation": "product_category_name"
}

# Initialize dictionary to store duplicate analysis results
duplicate_summary = {}

# Iterate through each dataset and analyze duplicates
for dataset_name, df in datasets.items():
    print(f"\n📊 Dataset: {dataset_name}")
    print(f"   Shape: {df.shape}")
    
    # Retrieve the primary key column for this dataset
    key_col = key_columns.get(dataset_name)
    
    # Check if key column exists in the dataset
    if key_col and key_col in df.columns:
        # Calculate total rows and unique values in the key column
        total_rows = len(df)
        unique_count = df[key_col].nunique()
        duplicate_count = total_rows - unique_count
        
        # Find all duplicate values and their frequencies
        duplicates = df[df.duplicated(subset=[key_col], keep=False)][key_col].value_counts()
        
        # Store the analysis results in a summary dictionary
        duplicate_summary[dataset_name] = {
            "key_column": key_col,
            "total_rows": total_rows,
            "unique_values": unique_count,
            "duplicate_rows": duplicate_count,
            "duplicate_percentage": (duplicate_count / total_rows * 100) if total_rows > 0 else 0
        }
        
        # Print detailed duplicate analysis for this dataset
        print(f"   Key Column: {key_col}")
        print(f"   Total Rows: {total_rows}")
        print(f"   Unique Values: {unique_count}")
        print(f"   Duplicate Rows: {duplicate_count} ({duplicate_summary[dataset_name]['duplicate_percentage']:.2f}%)")
        
        # Display top 5 duplicate values if any exist
        if duplicate_count > 0:
            print(f"   Top Duplicates in '{key_col}':")
            print(f"   {duplicates.head(5).to_dict()}")
    else:
        # Handle case where key column doesn't exist in dataset
        print(f"   ⚠️  Could not find key column '{key_col}' in dataset")
        print(f"   Available columns: {list(df.columns)}")

# Create a summary table of all duplicate analysis results
print("\n" + "=" * 70)
print("SUMMARY TABLE")
print("=" * 70)
summary_df = pd.DataFrame(duplicate_summary).T
print(summary_df)

DUPLICATE ANALYSIS FOR EACH DATASET

📊 Dataset: olist_sellers_dataset
   Shape: (3095, 4)
   Key Column: seller_id
   Total Rows: 3095
   Unique Values: 3095
   Duplicate Rows: 0 (0.00%)

📊 Dataset: product_category_name_translation
   Shape: (71, 2)
   Key Column: product_category_name
   Total Rows: 71
   Unique Values: 71
   Duplicate Rows: 0 (0.00%)

📊 Dataset: olist_orders_dataset
   Shape: (99441, 8)
   Key Column: order_id
   Total Rows: 99441
   Unique Values: 99441
   Duplicate Rows: 0 (0.00%)

📊 Dataset: olist_order_items_dataset
   Shape: (112650, 7)
   ⚠️  Could not find key column 'None' in dataset
   Available columns: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

📊 Dataset: olist_customers_dataset
   Shape: (99441, 5)
   Key Column: customer_id
   Total Rows: 99441
   Unique Values: 99441
   Duplicate Rows: 0 (0.00%)

📊 Dataset: olist_geolocation_dataset
   Shape: (1000163, 5)
   ⚠️  Could not find key column '